# Notebook Ordenado del TFM

Cuaderno maestro para la defensa del portal hipotecario.

Este notebook deja separadas las responsabilidades metodologicas:

- `tfm_reorganizado.ipynb`: fuente de las proyecciones macroeconomicas futuras.
- `data_science_provincial.ipynb`: fuente del modelo de demanda hipotecaria, con `RF_Nac` para el total nacional y `RF` provincial para territorios.
- `HIPOTECAS_PROV.ipynb`: fuente del pricing hipotecario personalizado.
- `Web/demanda_models.py`: implementacion reusable en Flask de los modelos de demanda usados por la web.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

ROOT = Path.cwd()
WEB_DIR = ROOT / 'Web'
sys.path.insert(0, str(WEB_DIR))

from demanda_models import build_demanda_response, _train_artifacts

DATASET_PATH = ROOT / 'dataset_final_tfm.csv'
PRED_PATH = ROOT / 'predicciones_provinciales_2026_2030.csv'
TFM_REORG_PATH = ROOT / 'notebooks acabados' / 'tfm_reorganizado.ipynb'
DATA_SCIENCE_PATH = ROOT / 'data_science_provincial.ipynb'

df = pd.read_csv(DATASET_PATH)
df_preds = pd.read_csv(PRED_PATH)

print(f'Dataset historico: {df.shape[0]:,} filas | {df["Provincia"].nunique()} provincias')
print(f'Predicciones macro CSV: {df_preds.shape[0]:,} filas | {df_preds["provincia"].nunique()} provincias')
print('Variables CSV:', sorted(df_preds['variable'].unique()))

## 1. Verificacion de procedencia del CSV macro

La comprobacion contrasta el CSV `predicciones_provinciales_2026_2030.csv` contra la logica macro nacional documentada en `tfm_reorganizado.ipynb`.

Nota metodologica: si el CSV es provincial, la comparacion razonable contra un notebook nacional es:

- medias provinciales para `PIB_pc`, `Tasa_Paro`, `Precio_m2` y `tasa_ahorro`;
- suma provincial para `Transmisiones_Total`.

In [ ]:
def generar_macro_tfm_reorganizado(df):
    df = df.copy()
    df['Trimestre'] = pd.to_datetime(df['Trimestre'])

    all_cols = ['PIB_pc', 'Tasa_Paro', 'Precio_m2', 'tasa_ahorro', 'Euribor', 'IPC']
    targets = ['PIB_pc', 'Tasa_Paro', 'Precio_m2', 'tasa_ahorro']
    escenarios_bce = {
        'Euribor': {'2026': 2.00, '2027': 2.10, '2028': 2.30, '2029': 2.15, '2030': 2.15, '2031': 2.15},
        'IPC': {'2026': 1.88, '2027': 2.10, '2028': 2.00, '2029': 2.00, '2030': 2.00, '2031': 2.00},
    }

    def anuales_a_trimestres(d):
        vals = []
        for anio in sorted(d.keys()):
            vals.extend([d[anio]] * 4)
        return vals

    n_steps = len(anuales_a_trimestres(escenarios_bce['Euribor']))
    df_nac = df.groupby('Trimestre')[all_cols].mean().sort_index()
    df_full = df_nac.dropna(subset=targets)
    future_idx = pd.date_range(df_full.index[-1] + pd.DateOffset(months=3), periods=n_steps, freq='QS')

    trans_nac = df.groupby('Trimestre')['Transmisiones_Total'].sum().sort_index().dropna()
    trans_clean = trans_nac.iloc[:-1]

    pib = sm.tsa.SARIMAX(pd.Series(df_full['PIB_pc'].values), order=(1, 1, 0), seasonal_order=(1, 1, 0, 4)).fit(disp=False).get_forecast(n_steps).predicted_mean.values

    paro_ts = pd.Series(df_full['Tasa_Paro'].values)
    best_aic, best_res = np.inf, None
    for p in range(4):
        for d in range(2):
            for q in range(3):
                try:
                    res = sm.tsa.ARIMA(paro_ts, order=(p, d, q)).fit()
                    if res.aic < best_aic:
                        best_aic, best_res = res.aic, res
                except Exception:
                    pass
    paro = np.clip(best_res.get_forecast(n_steps).predicted_mean.values, 8.5, None)

    precio_ts = df_full['Precio_m2'].dropna()
    tendencia = precio_ts.diff().iloc[-8:].mean()
    base = np.array([precio_ts.iloc[-(4 - i % 4)] for i in range(n_steps)])
    drift = np.array([(i // 4 + 1) * 4 * tendencia for i in range(n_steps)])
    precio = base + drift

    ahorro_ts = pd.Series(df_full['tasa_ahorro'].values)
    best_aic, best_res = np.inf, None
    for p in range(4):
        for d in range(2):
            for q in range(3):
                try:
                    res = sm.tsa.ARIMA(ahorro_ts, order=(p, d, q)).fit()
                    if res.aic < best_aic:
                        best_aic, best_res = res.aic, res
                except Exception:
                    pass
    ahorro = best_res.get_forecast(n_steps).predicted_mean.values

    tendencia_trans = trans_clean.diff(4).iloc[-8:].mean()
    base_trans = np.array([float(trans_clean.iloc[-(4 - i % 4)]) for i in range(n_steps)])
    drift_trans = np.array([(i // 4 + 1) * tendencia_trans for i in range(n_steps)])
    transmisiones = base_trans + drift_trans

    out = pd.DataFrame(index=future_idx)
    out['PIB_pc'] = pib
    out['Tasa_Paro'] = paro
    out['Precio_m2'] = precio
    out['tasa_ahorro'] = ahorro
    out['Transmisiones_Total'] = transmisiones
    return out.loc['2026':'2030']

macro_tfm = generar_macro_tfm_reorganizado(df)

df_preds_cmp = df_preds.copy()
df_preds_cmp['date'] = pd.to_datetime(df_preds_cmp['trimestre'])

rows = []
for var in ['PIB_pc', 'Tasa_Paro', 'Precio_m2', 'tasa_ahorro', 'Transmisiones_Total']:
    sub = df_preds_cmp[df_preds_cmp['variable'] == var]
    if sub.empty:
        rows.append({'variable': var, 'estado': 'NO ESTA EN CSV'})
        continue
    agg = 'sum' if var == 'Transmisiones_Total' else 'mean'
    csv_q = sub.groupby('date')['valor'].agg(agg).sort_index()
    aligned = pd.concat([macro_tfm[var].rename('tfm_reorganizado'), csv_q.rename('csv')], axis=1).dropna()
    diff = (aligned['tfm_reorganizado'] - aligned['csv']).abs()
    pct = diff / aligned['tfm_reorganizado'].abs().replace(0, np.nan)
    rows.append({
        'variable': var,
        'estado': 'COINCIDE' if diff.max() < 1e-3 else 'DIFIERE',
        'max_abs_diff': round(float(diff.max()), 4),
        'mean_abs_pct_diff': round(float(pct.mean() * 100), 4),
        'tfm_first': round(float(aligned['tfm_reorganizado'].iloc[0]), 4),
        'csv_first': round(float(aligned['csv'].iloc[0]), 4),
    })

check_csv = pd.DataFrame(rows)
check_csv

## 2. Demanda hipotecaria: modelos correctos

El modelo operativo de demanda procede de `data_science_provincial.ipynb`:

- Nacional: `RF_Nac`, `RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=3, max_features='sqrt')`.
- Provincial: `RF`, `RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=5, max_features='sqrt')`.

La web importa estas reglas desde `Web/demanda_models.py` para evitar divergencias entre notebook y Flask.

In [ ]:
art = _train_artifacts(df, df_preds)

print('Modelo nacional : RF_Nac')
print('MAPE nacional   :', art['nac_mape'], '%')
print('R2 nacional     :', art['nac_r2'])
print('Features nacional:', len(art['features_nac']))
print()
print('Modelo provincial : RF')
print('MAPE provincial   :', art['prov_mape'], '%')
print('R2 provincial     :', art['prov_r2'])
print('Features provincial:', len(art['features_prov']))

## 3. Forecast y stress test usados por la web

Ejemplos equivalentes a las llamadas de Flask. El stress test se calcula sobre el trimestre seleccionado, con escenarios relativos al Euribor base de ese año/trimestre.

In [ ]:
casos = [
    ('Nacional 2026-Q1', '2026-Q1', ''),
    ('Madrid 2026-Q1', '2026-Q1', 'Madrid'),
    ('A Coruna 2027-Q2', '2027-Q2', 'A Coruña'),
    ('Nacional 2030-Q4', '2030-Q4', ''),
]

for nombre, year, prov in casos:
    res, status = build_demanda_response(df, df_preds, year, prov)
    print('
' + '=' * 72)
    print(nombre, '| HTTP equivalente:', status)
    print('KPIs:', res['kpis'])
    print('Forecast 4 primeros:', list(zip(res['forecast']['labels'][:4], res['forecast']['values'][:4])))
    print('Stress:', res['stress_test'])

## 4. Conclusiones operativas

- La web debe usar `RF_Nac` cuando no se selecciona provincia.
- La web debe usar `RF` provincial cuando se selecciona una provincia.
- El CSV macro actual debe revisarse si se exige coincidencia estricta con `tfm_reorganizado.ipynb`: la comprobacion anterior muestra si coincide o difiere.
- `notebook_ordenado.ipynb` es el punto unico para presentar el flujo completo al tribunal.